## Task-1: Implement rotor machine <br>

In this task a classic rotor machine should be
implemented subject to the following conditions:
* Rotor machine should have 3 rotors
* Develop a code that simulates the working of rotor machine in such a way
that each rotor should rotate after character is encrypted.
* The developed system should include provisions for decryption process too.
* Test your rotor machine design using the following messages: ’HELLO’,
’HOPE’, and ’NEW YEAR’. Document your results in the format shown
in Table 2.

### Code

In [10]:
import random
import string

class RotorMachine: 

    @staticmethod
    def generate_random_rotor():
        alphabet = list(string.ascii_uppercase)
        random.shuffle(alphabet)
        return ''.join(alphabet)

    @staticmethod
    def generate_random_reflector():
        alphabet = list(string.ascii_uppercase)
        shuffled = alphabet[:]
        random.shuffle(shuffled)
        reflector = {}
        for a, b in zip(alphabet, shuffled):
            reflector[a] = b
            reflector[b] = a
        return reflector
    
    def rotors(self):
        rotor1 = self.generate_random_rotor()
        rotor2 = self.generate_random_rotor()
        rotor3 = self.generate_random_rotor()
        reflector = self.generate_random_reflector()
        return rotor1, rotor2, rotor3, reflector
    
    def instance(self, rotor1, rotor2, rotor3, reflector):
        self.rotors = [rotor1, rotor2, rotor3]
        self.reflector = reflector
        self.rotor_positions = [0, 0, 0]
        self.encryption_positions = []
        return self

    def rotate_rotors(self):
        self.rotor_positions[0] = (self.rotor_positions[0] + 1) % 26
        if self.rotor_positions[0] == 0:
            self.rotor_positions[1] = (self.rotor_positions[1] + 1) % 26
            if self.rotor_positions[1] == 0:
                self.rotor_positions[2] = (self.rotor_positions[2] + 1) % 26

    def encrypt_char(self, char):
        index = ord(char) - 65  # Get index before rotation
        self.encryption_positions.append(list(self.rotor_positions)) # Record position BEFORE rotation
        self.rotate_rotors() # Rotate AFTER recording position
        for i, rotor in enumerate(self.rotors):
            index = (index + self.rotor_positions[i]) % 26
            index = (ord(rotor[index]) - 65) % 26
        index = ord(self.reflector[chr(index + 65)]) - 65
        for i, rotor in enumerate(reversed(self.rotors)):
            index = (rotor.index(chr(index + 65)) - self.rotor_positions[2 - i]) % 26
        return chr(index + 65)
    
    def encrypt_message(self, message):
        encrypted_message = ''
        for char in message:
            encrypted_message += self.encrypt_char(char)
        return encrypted_message
    
   

    def decrypt_message(self, encrypted_message):
        self.encryption_positions.reverse()
        decrypted_message = ''
        for char in encrypted_message:
            rotor_pos = self.encryption_positions.pop()
            index = ord(char) - 65

            # Corrected Reverse Pass: Subtract rotor position BEFORE finding index
            for i, rotor in enumerate(reversed(self.rotors)):
                index = (index - rotor_pos[2 - i] + 26) % 26  # Subtract position first
                index = rotor.index(chr(index + 65))

            index = ord(self.reflector[chr(index + 65)]) - 65

            # Corrected Reverse Pass: Subtract rotor position BEFORE finding index
            for i, rotor in enumerate(self.rotors):
                index = (index - rotor_pos[i] + 26) % 26  # Subtract position first
                index = rotor.index(chr(index + 65))

            decrypted_message += chr(index + 65)
        return decrypted_message


rotor1, rotor2, rotor3, reflector = RotorMachine().rotors()
print(rotor1)
print(rotor2)
print(rotor3)
print(reflector)

# Create rotor machine instance
rotor_machine = RotorMachine().instance(rotor1, rotor2, rotor3, reflector)

# Test messages
messages = ['HELLO', 'HOPE', 'NEW YEAR']
encrypted_messages = [rotor_machine.encrypt_message(message) for message in messages]
decrypted_messages = [rotor_machine.decrypt_message(encrypted_message) for encrypted_message in encrypted_messages]

# Print results
for message, encrypted_message, decrypted_message in zip(messages, encrypted_messages, decrypted_messages):
    print(f'Original: {message} -> Encrypted:{encrypted_message} -> Decrypted:{decrypted_message}')



## Task-2: Implementing DES

In this task DES scheme should be implemented. Test
the operation of the developed DES scheme using the same three original messages
given in Task-1, and document the results once again similar to Table 1.

```
from Crypto.Random import get_random_bytes
from Crypto.Cipher import DES
from Crypto.Util.Padding import pad, unpad
import pandas as pd

class DESMachine:

    @staticmethod
    def generate_random_key():
        # Generate a random 8-byte key for DES
        return get_random_bytes(8)

    def rotor_machine_encrypt(text, shift=3):
        return ''.join([chr((ord(char) + shift) % 256) for char in text])
    
    def rotor_machine_decrypt(text, shift=3):	
        return ''.join([chr((ord(char) - shift) % 256) for char in text])

    def des_encrypt(text, key):
        des = DES.new(key, DES.MODE_ECB)
        padded_text = pad(text.encode(), DES.block_size)
        encrypted_text = des.encrypt(padded_text)
        return encrypted_text
    
    def des_decrypt(encrypted_text, key):
        des = DES.new(key, DES.MODE_ECB)
        decrypted_text = des.decrypt(encrypted_text)
        return unpad(decrypted_text, DES.block_size).decode()

des_key = DESMachine.generate_random_key()
print(f"DesKey:{des_key}")

messages = ['HELLO', 'HOPE', 'NEW YEAR']

results = {
    "Notation": ["M", "E1", "E2", "D1", "D2"],
    "Description": ["Original Message", "First encryption using rotor machine", 
                    "Second encryption using DES", "First decryption using DES", "Second decryption using rotor machine"]
}
table_data = []

for message in messages:
    M = message
    E1 = DESMachine.rotor_machine_encrypt(M)
    E2 = DESMachine.des_encrypt(E1, des_key)
    D1 = DESMachine.des_decrypt(E2, des_key)
    D2 = DESMachine.rotor_machine_decrypt(D1)
    table_data.append([M, E1, E2.hex(), D1, D2])

df = pd.DataFrame(table_data, columns=results["Notation"])
df
```

### Code

In [33]:
from Crypto.Random import get_random_bytes
from Crypto.Cipher import DES
from Crypto.Util.Padding import pad, unpad
import pandas as pd

class DESMachine:

    @staticmethod
    def generate_random_key():
        # Generate a random 8-byte key for DES
        return get_random_bytes(8)

    def rotor_machine_encrypt(text, shift=3):
        return ''.join([chr((ord(char) + shift) % 256) for char in text])
    
    def rotor_machine_decrypt(text, shift=3):	
        return ''.join([chr((ord(char) - shift) % 256) for char in text])

    def des_encrypt(text, key):
        des = DES.new(key, DES.MODE_ECB)
        padded_text = pad(text.encode(), DES.block_size)
        encrypted_text = des.encrypt(padded_text)
        return encrypted_text
    
    def des_decrypt(encrypted_text, key):
        des = DES.new(key, DES.MODE_ECB)
        decrypted_text = des.decrypt(encrypted_text)
        return unpad(decrypted_text, DES.block_size).decode()

des_key = DESMachine.generate_random_key()
print(f"DesKey:{des_key}")

messages = ['HELLO', 'HOPE', 'NEW YEAR']

results = {
    "Notation": ["M", "E1", "E2", "D1", "D2"],
    "Description": ["Original Message", "First encryption using rotor machine", 
                    "Second encryption using DES", "First decryption using DES", "Second decryption using rotor machine"]
}
table_data = []

for message in messages:
    M = message
    E1 = DESMachine.rotor_machine_encrypt(M)
    E2 = DESMachine.des_encrypt(E1, des_key)
    D1 = DESMachine.des_decrypt(E2, des_key)
    D2 = DESMachine.rotor_machine_decrypt(D1)
    table_data.append([M, E1, E2.hex(), D1, D2])

df = pd.DataFrame(table_data, columns=results["Notation"])
df

DesKey:b'\xaeW^\x98\x0b\xbeu\xa4'


,M,E1,E2,D1,D2
0,HELLO,KHOOR,c1af2c8d4948ef0c,KHOOR,HELLO
1,HOPE,KRSH,3d6cec1dc57f2a3e,KRSH,HOPE
2,NEW YEAR,QHZ#\HDU,4bb3ff9d6958e5c0684cefe9910043b9,QHZ#\HDU,NEW YEAR


## Task-3: Implement hybrid Cryptographic system

In this task two layer encryption
will be done. First, M will be passed through rotor machine to get E1,
which then should be passed through DES to get E2. In the similar manner
decryption process should be done. Test the developed hybrid system on the
following messages, and document the results as shown in Table 3

```
from Crypto.Random import get_random_bytes
from Crypto.Cipher import DES
from Crypto.Util.Padding import pad, unpad
import string
import pandas as pd

class HybridCryptographisSystem:

    @staticmethod
    def generate_random_key():
        # Generate a random key of the specified length
        return get_random_bytes(8)
    
    @staticmethod
    def rotor_machine_encrypt(message, shift=3):
        # Encrypt the message using a simple Caesar cipher with a shift
        encrypted_message = ''
        for char in message:
            if char in string.ascii_uppercase:
                encrypted_message += chr((ord(char) - 65 + shift) % 26 + 65)
            elif char in string.ascii_lowercase:
                encrypted_message += chr((ord(char) - 97 + shift) % 26 + 97)
            else:
                encrypted_message += char
        return encrypted_message
    
    @staticmethod
    def rotor_machine_decrypt(encrypted_message, shift=3):
        # Decrypt the message using a simple Caesar cipher with a shift
        decrypted_message = ''
        for char in encrypted_message:
            if char in string.ascii_uppercase:
                decrypted_message += chr((ord(char) - 65 - shift) % 26 + 65)
            elif char in string.ascii_lowercase:
                decrypted_message += chr((ord(char) - 97 - shift) % 26 + 97)
            else:
                decrypted_message += char
        return decrypted_message
    
    @staticmethod
    def des_encrypt(key, message):
        # Encrypt the message using DES encryption
        cipher = DES.new(key, DES.MODE_ECB)
        padded_message = pad(message.encode(), DES.block_size)
        encrypted_message = cipher.encrypt(padded_message)
        return encrypted_message
    
    @staticmethod
    def des_decrypt(key, encrypted_message):
        # Decrypt the message using DES decryption
        cipher = DES.new(key, DES.MODE_ECB)
        unpadded_message = unpad(cipher.decrypt(encrypted_message), DES.block_size)
        return unpadded_message.decode('utf-8')
    
# Test
des_key = HybridCryptographisSystem.generate_random_key()
print(f"DesKey:{des_key}")

messages = ["HOW ARE YOU", "HAPPY NEW YEAR", "WELCOME TO PUERTO RICO"]
results = []

for message in messages:
    # Encrypt the message using the rotor machine
    E1 = HybridCryptographisSystem.rotor_machine_encrypt(message)
    # Encrypt the rotor machine output using DES
    E2 = HybridCryptographisSystem.des_encrypt(des_key, E1)
    # Decrypt the DES output
    E3 = HybridCryptographisSystem.des_decrypt(des_key, E2)
    # Decrypt the rotor machine output
    M_decrypted = HybridCryptographisSystem.rotor_machine_decrypt(E3)

    results.append({
        'M': message,
        'E1': E1,
        'E2': E2.hex(),
        'D1': HybridCryptographisSystem.des_decrypt(des_key, E2),
        'D2': HybridCryptographisSystem.rotor_machine_decrypt(E3),
    })

# Create a DataFrame to display the results
df = pd.DataFrame(results)
df.columns = ['Message (M)', 'Rotor Machine (E1)', 'DES Encryption (E2)', 'DES Decryption (D1)', 'Rotor Machine Decryption (D2)']
df
```

### Code

In [20]:
from Crypto.Random import get_random_bytes
from Crypto.Cipher import DES
from Crypto.Util.Padding import pad, unpad
import string
import pandas as pd

class HybridCryptographisSystem:

    @staticmethod
    def generate_random_key():
        # Generate a random key of the specified length
        return get_random_bytes(8)
    
    @staticmethod
    def rotor_machine_encrypt(message, shift=3):
        # Encrypt the message using a simple Caesar cipher with a shift
        encrypted_message = ''
        for char in message:
            if char in string.ascii_uppercase:
                encrypted_message += chr((ord(char) - 65 + shift) % 26 + 65)
            elif char in string.ascii_lowercase:
                encrypted_message += chr((ord(char) - 97 + shift) % 26 + 97)
            else:
                encrypted_message += char
        return encrypted_message
    
    @staticmethod
    def rotor_machine_decrypt(encrypted_message, shift=3):
        # Decrypt the message using a simple Caesar cipher with a shift
        decrypted_message = ''
        for char in encrypted_message:
            if char in string.ascii_uppercase:
                decrypted_message += chr((ord(char) - 65 - shift) % 26 + 65)
            elif char in string.ascii_lowercase:
                decrypted_message += chr((ord(char) - 97 - shift) % 26 + 97)
            else:
                decrypted_message += char
        return decrypted_message
    
    @staticmethod
    def des_encrypt(key, message):
        # Encrypt the message using DES encryption
        cipher = DES.new(key, DES.MODE_ECB)
        padded_message = pad(message.encode(), DES.block_size)
        encrypted_message = cipher.encrypt(padded_message)
        return encrypted_message
    
    @staticmethod
    def des_decrypt(key, encrypted_message):
        # Decrypt the message using DES decryption
        cipher = DES.new(key, DES.MODE_ECB)
        unpadded_message = unpad(cipher.decrypt(encrypted_message), DES.block_size)
        return unpadded_message.decode('utf-8')
    
# Test
des_key = HybridCryptographisSystem.generate_random_key()
print(f"DesKey:{des_key}")

messages = ["HOW ARE YOU", "HAPPY NEW YEAR", "WELCOME TO PUERTO RICO"]
results = []

for message in messages:
    # Encrypt the message using the rotor machine
    E1 = HybridCryptographisSystem.rotor_machine_encrypt(message)
    # Encrypt the rotor machine output using DES
    E2 = HybridCryptographisSystem.des_encrypt(des_key, E1)
    # Decrypt the DES output
    E3 = HybridCryptographisSystem.des_decrypt(des_key, E2)
    # Decrypt the rotor machine output
    M_decrypted = HybridCryptographisSystem.rotor_machine_decrypt(E3)

    results.append({
        'M': message,
        'E1': E1,
        'E2': E2.hex(),
        'D1': HybridCryptographisSystem.des_decrypt(des_key, E2),
        'D2': HybridCryptographisSystem.rotor_machine_decrypt(E3),
    })

# Create a DataFrame to display the results
df = pd.DataFrame(results)
df.columns = ['Message (M)', 'Rotor Machine (E1)', 'DES Encryption (E2)', 'DES Decryption (D1)', 'Rotor Machine Decryption (D2)']
df

DesKey:b'{\x12\xd6Q\x8a\x1d-}'


,Message (M),Rotor Machine (E1),DES Encryption (E2),DES Decryption (D1),Rotor Machine Decryption (D2)
0,HOW ARE YOU,KRZ DUH BRX,949cc18cc89449060f04c9237675c9e7,KRZ DUH BRX,HOW ARE YOU
1,HAPPY NEW YEAR,KDSSB QHZ BHDU,c7b24f6a15c472a8f08a6e9115c4f225,KDSSB QHZ BHDU,HAPPY NEW YEAR
2,WELCOME TO PUERTO RICO,ZHOFRPH WR SXHUWR ULFR,fe5d3006fd19f307f07f7558d97d6d2e2a62c591f4ea6d8b,ZHOFRPH WR SXHUWR ULFR,WELCOME TO PUERTO RICO
